# Model evaluation + figures (AlphaEarth)

This notebook evaluates the trained Ridge and XGBoost models from `../outputs/models/checkpoints` and produces the main evaluation figures.

Split out of the prior `01_model_evaluation.ipynb`.

In [ ]:
# Run from `alphaearth_src/notebooks` so relative paths resolve.
# Requires `python scripts/modeling/train_models.py` to have produced checkpoints.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import joblib
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, median_absolute_error
from IPython.display import display

# Publication-ish theme: light major y-grid only, no minor grid
sns.set_theme(
    style="white",
    font_scale=1.05,
    rc={
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "0.88",
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
    },
)

CHECKPOINT_DIR = "../outputs/models/checkpoints"


## 1. Load data and reconstruct filtered targets

Merge embeddings CSV with the same filters as training (nonzero ln targets).


In [ ]:
print("Loading data...")
file_path = "../data/embeddings/00_all_mexico_embeddings_combined.csv"
df = pd.read_csv(file_path)

state_dummies = pd.get_dummies(df["CVE_ENT"], prefix="ENT", drop_first=True)
embedding_cols = [f"A{i:02d}" for i in range(64)]

X_base = df[embedding_cols]
# Columns log_POBTOT / log_popden store natural log (ln); back-transform with np.exp(...)
y_log_pobtot = df["log_POBTOT"]
y_log_popden = df["log_popden"]

X_base = X_base.replace([np.inf, -np.inf], np.nan).fillna(0)
y_log_pobtot = y_log_pobtot.replace([np.inf, -np.inf], np.nan).fillna(0)
y_log_popden = y_log_popden.replace([np.inf, -np.inf], np.nan).fillna(0)

# Filter invalid / placeholder rows (ln targets masked as 0)
mask = (y_log_popden != 0) & (y_log_pobtot != 0)

X_filt = X_base[mask].copy()
state_dummies_filt = state_dummies[mask].copy()

# Combine embeddings and state dummies
X = pd.concat([X_filt, state_dummies_filt], axis=1)
y_den = y_log_popden[mask].copy()

print(f"Dataset size after filtering: {len(X):,} rows")
print(f"Number of features: {X.shape[1]}")


## 2. Aggregate out-of-fold predictions

Load K-fold checkpoints from `checkpoints/` and fill OOF prediction vectors.


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds_ridge = np.zeros(len(X))
oof_preds_xgb = np.zeros(len(X))

xgb_importances = []

fold = 1
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y_den.iloc[train_index], y_den.iloc[test_index]

    scaler = joblib.load(f"{CHECKPOINT_DIR}/scaler_fold{fold}.pkl")
    ridge = joblib.load(f"{CHECKPOINT_DIR}/ridge_fold{fold}.pkl")

    xgb_model = xgb.XGBRegressor()
    xgb_model.load_model(f"{CHECKPOINT_DIR}/xgb_fold{fold}.json")

    X_test_sc = scaler.transform(X_test)
    oof_preds_ridge[test_index] = ridge.predict(X_test_sc)
    oof_preds_xgb[test_index] = xgb_model.predict(X_test)

    xgb_importances.append(xgb_model.feature_importances_)

    fold += 1

print("Out-of-fold predictions gathered for all 5 folds.")


## 3. Overall performance metrics

Ridge vs XGBoost on ln scale and back-transformed density.


In [ ]:
y_den_orig = np.exp(y_den)
oof_preds_ridge_orig = np.exp(oof_preds_ridge)
oof_preds_xgb_orig = np.exp(oof_preds_xgb)

results = pd.DataFrame({
    "Model": ["Ridge", "XGBoost"],
    "R² (ln)": [r2_score(y_den, oof_preds_ridge), r2_score(y_den, oof_preds_xgb)],
    "RMSE (ln)": [np.sqrt(mean_squared_error(y_den, oof_preds_ridge)), np.sqrt(mean_squared_error(y_den, oof_preds_xgb))],
    "MAE (ln)": [mean_absolute_error(y_den, oof_preds_ridge), mean_absolute_error(y_den, oof_preds_xgb)],
    "R² (orig scale)": [r2_score(y_den_orig, oof_preds_ridge_orig), r2_score(y_den_orig, oof_preds_xgb_orig)],
    "MAE (orig scale)": [mean_absolute_error(y_den_orig, oof_preds_ridge_orig), mean_absolute_error(y_den_orig, oof_preds_xgb_orig)],
    "Median AE (orig scale)": [median_absolute_error(y_den_orig, oof_preds_ridge_orig), median_absolute_error(y_den_orig, oof_preds_xgb_orig)],
})
display(results.style.format({"R² (ln)": "{:.4f}", "RMSE (ln)": "{:.4f}", "MAE (ln)": "{:.4f}",
                              "R² (orig scale)": "{:.4f}", "MAE (orig scale)": "{:,.1f}", "Median AE (orig scale)": "{:,.1f}"})
        .background_gradient(subset=["R² (ln)", "R² (orig scale)"], cmap="Greens"))


## 4. Scatter plots: actual vs predicted

Four panels (ln and original scale); each title shows R²(ln) and R²(orig).


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

r2_ridge_ln = r2_score(y_den, oof_preds_ridge)
r2_ridge_o = r2_score(y_den_orig, oof_preds_ridge_orig)
r2_xgb_ln = r2_score(y_den, oof_preds_xgb)
r2_xgb_o = r2_score(y_den_orig, oof_preds_xgb_orig)

axes[0, 0].scatter(y_den, oof_preds_ridge, alpha=0.1, s=2, c="#4C72B0", rasterized=True)
axes[0, 0].plot([y_den.min(), y_den.max()], [y_den.min(), y_den.max()], "r--", lw=2)
axes[0, 0].set_xlabel("Actual ln(popden)")
axes[0, 0].set_ylabel("Predicted ln(popden)")
axes[0, 0].set_title(
    f"Ridge — ln scale\nR²(ln)={r2_ridge_ln:.3f}   R²(orig)={r2_ridge_o:.3f}",
    fontsize=11,
)

axes[0, 1].scatter(y_den, oof_preds_xgb, alpha=0.1, s=2, c="#55A868", rasterized=True)
axes[0, 1].plot([y_den.min(), y_den.max()], [y_den.min(), y_den.max()], "r--", lw=2)
axes[0, 1].set_xlabel("Actual ln(popden)")
axes[0, 1].set_ylabel("Predicted ln(popden)")
axes[0, 1].set_title(
    f"XGBoost — ln scale\nR²(ln)={r2_xgb_ln:.3f}   R²(orig)={r2_xgb_o:.3f}",
    fontsize=11,
)

lo, hi = y_den_orig.min(), y_den_orig.max()
axes[1, 0].scatter(y_den_orig, oof_preds_ridge_orig, alpha=0.08, s=2, c="#4C72B0", rasterized=True)
axes[1, 0].plot([lo, hi], [lo, hi], "r--", lw=2)
axes[1, 0].set_xlabel("Actual pop. density (orig. scale)")
axes[1, 0].set_ylabel("Predicted pop. density (orig. scale)")
axes[1, 0].set_title(
    f"Ridge — original scale\nR²(ln)={r2_ridge_ln:.3f}   R²(orig)={r2_ridge_o:.3f}",
    fontsize=11,
)

axes[1, 1].scatter(y_den_orig, oof_preds_xgb_orig, alpha=0.08, s=2, c="#55A868", rasterized=True)
axes[1, 1].plot([lo, hi], [lo, hi], "r--", lw=2)
axes[1, 1].set_xlabel("Actual pop. density (orig. scale)")
axes[1, 1].set_ylabel("Predicted pop. density (orig. scale)")
axes[1, 1].set_title(
    f"XGBoost — original scale\nR²(ln)={r2_xgb_ln:.3f}   R²(orig)={r2_xgb_o:.3f}",
    fontsize=11,
)

plt.suptitle(
    "Out-of-fold actual vs predicted (ln and original scale; both R² on each panel)",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.show()


## 5. Feature importances (XGBoost)

Mean gain-based importance across the five folds; top 20 features.


In [ ]:
avg_importances = np.mean(xgb_importances, axis=0)
sorted_idx = np.argsort(avg_importances)

top_k = 20
top_idx = sorted_idx[-top_k:]
features = np.array(X.columns)

r2_xgb_ln_fi = r2_score(y_den, oof_preds_xgb)
r2_xgb_orig_fi = r2_score(y_den_orig, oof_preds_xgb_orig)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(top_k), avg_importances[top_idx], color="#C44E52", edgecolor="white", linewidth=0.5)
ax.set_yticks(range(top_k))
ax.set_yticklabels(features[top_idx], fontsize=10)
ax.set_xlabel("Average Feature Importance (Gain) across 5 Folds")
ax.set_title(f"Top {top_k} Most Important Features (XGBoost)", fontsize=13, fontweight="bold")
ax.text(
    0.99,
    0.02,
    f"OOF R²  ln={r2_xgb_ln_fi:.3f}  |  orig={r2_xgb_orig_fi:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=10,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#ccc", alpha=0.9),
)

plt.tight_layout()
plt.show()


## 6. State-wise performance (XGBoost OOF)

Per-state metrics for geographic consistency.


In [ ]:
state_names = {
    1: "AGS", 2: "BC", 3: "BCS", 4: "CAM", 5: "COAH", 6: "COL", 7: "CHIS",
    8: "CHIH", 9: "CDMX", 10: "DGO", 11: "GTO", 12: "GRO", 13: "HGO",
    14: "JAL", 15: "MEX", 16: "MICH", 17: "MOR", 18: "NAY", 19: "NL",
    20: "OAX", 21: "PUE", 22: "QRO", 23: "QROO", 24: "SLP", 25: "SIN",
    26: "SON", 27: "TAB", 28: "TAMPS", 29: "TLAX", 30: "VER", 31: "YUC", 32: "ZAC",
}

states_filtered = df.loc[mask, "CVE_ENT"].map(state_names)

state_metrics = []
for state in sorted(states_filtered.unique()):
    s_mask = (states_filtered == state)
    y_true_s = y_den[s_mask]
    y_pred_s = oof_preds_xgb[s_mask]

    y_true_s_orig = np.exp(y_true_s)
    y_pred_s_orig = np.exp(y_pred_s)

    n = s_mask.sum()
    if n < 5:
        continue

    state_metrics.append({
        "State": state,
        "N": n,
        "R² (ln)": r2_score(y_true_s, y_pred_s),
        "RMSE (ln)": np.sqrt(mean_squared_error(y_true_s, y_pred_s)),
        "MAE (ln)": mean_absolute_error(y_true_s, y_pred_s),
        "Mean Residual (ln)": np.mean(y_true_s - y_pred_s),
        "R² (orig)": r2_score(y_true_s_orig, y_pred_s_orig),
        "MAE (orig)": mean_absolute_error(y_true_s_orig, y_pred_s_orig),
        "Median AE (orig)": median_absolute_error(y_true_s_orig, y_pred_s_orig),
    })

state_df = pd.DataFrame(state_metrics).set_index("State").sort_values("R² (ln)", ascending=False)
display(state_df.style.format({
    "R² (ln)": "{:.3f}", "RMSE (ln)": "{:.3f}", "MAE (ln)": "{:.3f}",
    "Mean Residual (ln)": "{:+.3f}", "N": "{:,}",
    "R² (orig)": "{:.3f}", "MAE (orig)": "{:,.1f}", "Median AE (orig)": "{:,.1f}",
}).background_gradient(subset=["R² (ln)", "R² (orig)"], cmap="RdYlGn", vmin=0.3, vmax=0.85))


## 7. State-wise metric plots

Horizontal bars for R²(ln), R²(orig), and MAE(ln) with medians.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 10), sharey=True)

state_order = state_df.sort_values("R² (ln)", ascending=True).index
r2_ln_vals = state_df.loc[state_order, "R² (ln)"]
r2_orig_vals = state_df.loc[state_order, "R² (orig)"]
mae_vals = state_df.loc[state_order, "MAE (ln)"]

axes[0].barh(state_order, r2_ln_vals, color="#4C72B0", edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("R² (ln)")
axes[0].set_title("Per-state R² — ln scale", fontweight="bold")
axes[0].axvline(
    state_df["R² (ln)"].median(),
    color="k",
    ls="--",
    alpha=0.5,
    label=f"Median = {state_df['R² (ln)'].median():.3f}",
)
axes[0].legend(loc="lower right", fontsize=9)

axes[1].barh(state_order, r2_orig_vals, color="#2ca02c", edgecolor="white", linewidth=0.5)
axes[1].set_xlabel("R² (orig)")
axes[1].set_title("Per-state R² — original scale", fontweight="bold")
axes[1].axvline(
    state_df["R² (orig)"].median(),
    color="k",
    ls="--",
    alpha=0.5,
    label=f"Median = {state_df['R² (orig)'].median():.3f}",
)
axes[1].legend(loc="lower right", fontsize=9)

axes[2].barh(state_order, mae_vals, color="#C44E52", edgecolor="white", linewidth=0.5)
axes[2].set_xlabel("MAE (ln density)")
axes[2].set_title("Per-state MAE — ln scale", fontweight="bold")

plt.suptitle("XGBoost OOF — per state (ln vs orig R²)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## 8. Choropleth maps

Requires `../shps/complete/00a.shp` (or update the path). Side-by-side ln vs original R².


In [ ]:
import geopandas as gpd

print("Loading shapefile...")
try:
    gdf = gpd.read_file("../shps/complete/00a.shp")

    gdf["CVE_ENT"] = gdf["CVEGEO"].str[:2].astype(int)
    state_polygons = gdf.dissolve(by="CVE_ENT")

    state_abbr_to_id = {v: k for k, v in state_names.items()}
    state_df["CVE_ENT"] = state_df.index.map(state_abbr_to_id)

    state_polygons = state_polygons.merge(state_df, on="CVE_ENT", how="left")

    fig, axes = plt.subplots(1, 2, figsize=(20, 9))

    for ax in axes:
        state_polygons.plot(ax=ax, color="lightgrey", edgecolor="black", linewidth=0.5)

    valid_ln = state_polygons.dropna(subset=["R² (ln)"])
    valid_orig = state_polygons.dropna(subset=["R² (orig)"])

    valid_ln.plot(
        column="R² (ln)",
        ax=axes[0],
        legend=True,
        cmap="RdYlGn",
        edgecolor="black",
        linewidth=0.5,
        vmin=0.3,
        vmax=0.85,
        legend_kwds={"label": "R² (ln scale)", "orientation": "horizontal", "shrink": 0.75, "pad": 0.03},
    )
    axes[0].set_title("Out-of-fold R² — ln(pop. density)", fontsize=13, fontweight="bold")
    axes[0].axis("off")

    valid_orig.plot(
        column="R² (orig)",
        ax=axes[1],
        legend=True,
        cmap="RdYlGn",
        edgecolor="black",
        linewidth=0.5,
        vmin=0.3,
        vmax=0.85,
        legend_kwds={"label": "R² (original scale)", "orientation": "horizontal", "shrink": 0.75, "pad": 0.03},
    )
    axes[1].set_title("Out-of-fold R² — original scale", fontsize=13, fontweight="bold")
    axes[1].axis("off")

    plt.suptitle(
        "Geographic distribution of XGBoost OOF performance (ln vs original R²)",
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(
        f"Error loading or plotting shapefile: {e}\n"
        "Please ensure the path to your Mexico state boundaries shapefile is correct."
    )


## 9. Municipio-level evaluation (rollup vs direct)

**Roll-up:** polygon OOF ln-density predictions imply population `exp(pred)×area`; summed within each municipio (`CVE_ENT`×`CVE_MUN`) and divided by total municipio area gives municipio ln-density. Ground truth uses the same rollup from polygon targets `exp(log_popden)×area`, reported as `ln_density` in the rollup table, so it matches the training objective.

**Direct:** one row per municipio = area-weighted mean of the 64 embedding dimensions + state dummies; each fold’s Ridge/XGB checkpoint predicts ln-density and we average across folds. Interpret vs polygon-level calibration cautiously.

Run `python scripts/evaluation/municipio_evaluation.py` from the project root for the same metrics outside the notebook. Metrics JSON is written to `outputs/evaluation/municipio_evaluation.json` by default.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "scripts"))

from evaluation.municipio_evaluation import (
    evaluate_municipio,
    save_municipio_metrics_json,
)
import json

mun = evaluate_municipio(min_polygons=3)
summary_path = save_municipio_metrics_json(mun)
print(f"Saved municipio metrics JSON to: {summary_path}")
summary = {
    k: mun[k]
    for k in mun
    if k not in ("rollup_table", "direct_pred_ridge", "direct_pred_xgb")
}
print(json.dumps(summary, indent=2))

display(mun["rollup_table"].head(10))